# 03 · Reversible, same batch: 50M tokens, batch 32

The rule chosen in notebook 01, at exactly the baseline's batch, data order, schedule and seed. The only
difference from notebook 02 is how the residual stream moves between layers and what the backward pass
keeps. This is the like-for-like comparison: same work, what does reversibility cost in time and save in
memory?

In [1]:
import sys, os, json, pathlib, subprocess
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB and not pathlib.Path("ERA_V5").exists():
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/amukka/ERA_V5.git"], check=True)
    os.chdir("ERA_V5/session13_Distributed_Training")
ROOT = pathlib.Path.cwd()
while not (ROOT / "src").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
os.chdir(ROOT); sys.path.insert(0, str(ROOT))
if not (ROOT / "data" / "train.bin").exists():          # ~5 minutes: stream FineWeb-Edu, train the BPE
    subprocess.run([sys.executable, "tools/prepare_data.py"], check=True)

import torch, platform
from src import experiments as X
from src.train import pick_device
dev = pick_device()
print("device:", dev, "| torch", torch.__version__, "| python", platform.python_version(), "|", platform.platform())
if dev == "mps":
    print(f"MPS recommended working set: {torch.mps.recommended_max_memory()/2**30:.2f} GiB")
if dev == "cuda":
    print(torch.cuda.get_device_name(), f"{torch.cuda.get_device_properties(0).total_memory/2**30:.1f} GiB")
print(json.dumps(json.loads((ROOT / "data" / "meta.json").read_text()), indent=1))

device: mps | torch 2.13.0 | python 3.14.5 | macOS-26.6.2-arm64-arm-64bit-Mach-O
MPS recommended working set: 11.84 GiB
{
 "source": "HuggingFaceFW/fineweb-edu, config sample-10BT, streamed in order",
 "vocab_size": 8192,
 "eos_id": 0,
 "train_tokens": 56000000,
 "train_docs": 46080,
 "val_tokens": 1000000,
 "val_docs": 1024,
 "chars_per_token": 3.886,
 "note": "val.bin comes from the tokenizer's training documents; train.bin from the documents after them",
 "seconds": 314.3
}


**The model** (identical in every run): GPT, 10 layers, d_model 384, 6 heads, MLP 4x, sequence 512,
byte-level BPE vocabulary 8,192 with the head tied to the embedding. **21,053,184 parameters**, of which
17,710,848 are outside the embedding tables. No dropout anywhere, because a reversible backward pass
recomputes each block and a random mask would recompute differently. AdamW (0.9, 0.95, wd 0.1), peak LR
1e-3, 3% warmup then cosine to 10%, gradient clip 1.0, fp32.

In [2]:
chosen = json.loads((X.RESULTS / "chosen_variant.json").read_text())
print("rule:", chosen["variant"], "h =", chosen["h"])
rev = X.train(name=f"reversible_{chosen['variant']}_b32", variant=chosen["variant"], h=chosen["h"],
              batch=X.BASE_BATCH, tokens=X.TOKENS, eval_every=250, log_every=10)

rule: momentum h = 0.5


{"name": "reversible_momentum_b32", "device": "mps", "config": {"vocab_size": 8192, "seq_len": 512, "n_layer": 10, "n_head": 6, "d_model": 384, "variant": "momentum", "h": 0.5, "fp_iters": 6, "extra": {}}, "params": 21053184, "params_non_embedding": 17710848, "batch": 32, "tokens_per_step": 16384, "steps": 3051, "peak_lr": 0.001, "warmup": 91, "amp_bf16": 0, "torch": "2.13.0", "machine": "arm64", "processor": "arm"}


step     0/3051  tok   0.02M  loss 9.0825  gnorm 3.503  0 tok/s  0.0 min


memory audit at step 3: exact peak 2,755 MiB in aten._softmax_backward_data.default


step   100/3051  tok   1.64M  loss 6.2163  gnorm 0.401  8,947 tok/s  3.0 min


...

            eval @ 49.14M tokens: val loss 4.0264


step  3000/3051  tok  49.15M  loss 4.0640  gnorm 0.747  9,086 tok/s  91.8 min


step  3050/3051  tok  49.97M  loss 4.0007  gnorm 0.704  8,923 tok/s  93.3 min


            eval @ 49.97M tokens: val loss 4.0206


reversibility at final weights: grad rel err 9.58e-05, cosine 1.000000, state rebuild err 2.89e-05
{"tok_per_s": 8924.7, "final_train_loss": 4.000720977783203, "final_val_loss": 4.020622986555099, "peak_exact_mib": 2754.9, "peak_alloc_mib": 1620.7, "peak_driver_mib": 6211.1, "train_seconds": 5599.2}


In [3]:
keys = ["tokens_trained", "steps", "final_train_loss", "final_train_loss_avg_last5pct", "final_val_loss",
        "tok_per_s", "train_seconds", "peak_exact_mib", "peak_exact_op", "peak_alloc_mib", "peak_driver_mib"]
print(json.dumps({k: rev[k] for k in keys}, indent=1))
print(json.dumps(rev["reversibility"], indent=1))

{
 "tokens_trained": 49971200,
 "steps": 3051,
 "final_train_loss": 4.000720977783203,
 "final_train_loss_avg_last5pct": 4.01357962290446,
 "final_val_loss": 4.020622986555099,
 "tok_per_s": 8924.7,
 "train_seconds": 5599.2,
 "peak_exact_mib": 2754.9,
 "peak_exact_op": "aten._softmax_backward_data.default",
 "peak_alloc_mib": 1620.7,
 "peak_driver_mib": 6211.1
}
{
 "init": {
  "grad_rel_err": 2.200800561903834e-06,
  "grad_cosine": 0.9999999999972712,
  "state_rebuild_rel_err_per_layer": [
   5.324042831489351e-06,
   2.7300793590256944e-06,
   1.537154957986786e-06,
   9.912050700222608e-07,
   6.691512908219011e-07,
   5.152484163772897e-07,
   4.32310315545692e-07,
   2.8608965862986224e-07,
   2.3783076130712288e-07
  ],
  "state_rebuild_rel_err_max": 5.324042831489351e-06
 },
 "final": {
  "grad_rel_err": 9.583595126249242e-05,
  "grad_cosine": 0.999999995413234,
  "state_rebuild_rel_err_per_layer": [
   2.8914064387208782e-05,
   6.350663625198649e-06,
   4.238381279719761e-06,
 